In [26]:
!pip install openai-agents
!pip install openai
!pip install pydantic

In [27]:
from pydantic import BaseModel
from agents import Agent, ModelSettings, TResponseInputItem, Runner, RunConfig
from openai.types.shared.reasoning import Reasoning

class TriageRequestSchema(BaseModel):
  classification: str


class ApprovalAgentSchema(BaseModel):
  emailFrom: str
  defaultTo: str
  defaultSubject: str
  defaultBody: str


triage_request = Agent(
  name="Triage request",
  instructions="""Classify the user's request based on whether two documents have been provided recently in the conversation, and whether the user is asking a particular question.

If two documents are provided and there's no user question , respond with \"compare\".
If two documents are provided and there is a user question , respond with \"answer_question\".
If only one doc has been provided, or no docs have been provided, respond with \"request_upload\"""",
  model="gpt-4.1",
  output_type=TriageRequestSchema,
  model_settings=ModelSettings(
    temperature=1,
    top_p=1,
    max_tokens=2048,
    store=True
  )
)


propose_reconciliation = Agent(
  name="Propose reconciliation",
  instructions="Given the differences between the two documents, assemble a single option for how to reconcile the difference. If no order has been described, consider the first document the user's version and the second document the potential set of changes returned back to the user. The proposal you create will be sent to the user for approval.",
  model="gpt-5",
  model_settings=ModelSettings(
    store=True,
    reasoning=Reasoning(
      effort="minimal",
      summary="auto"
    )
  )
)


approval_agent = Agent(
  name="Approval agent",
  instructions="""Explain your approval reasoning. Help the user draft a proper response by filling out this data schema:

{
  emailFrom: 'user@test.com',
  defaultTo: 'user@test.com',
  defaultSubject: 'Document comparison proposal',
  defaultBody: \"Hey there, \n\nHope you're doing well! Just wanted to check in and see if there are any updates on the ChatKit roadmap. We're excited to see what's coming next and how we can make the most of the upcoming features.\n\nEspecially curious to see how you support widgets!\n\nBest,\",
}""",
  model="gpt-5-mini",
  output_type=ApprovalAgentSchema,
  model_settings=ModelSettings(
    store=True,
    reasoning=Reasoning(
      effort="low",
      summary="auto"
    )
  )
)


rejection_agent = Agent(
  name="Rejection agent",
  instructions="Explain your rejection reasoning.",
  model="gpt-5",
  model_settings=ModelSettings(
    store=True,
    reasoning=Reasoning(
      effort="low",
      summary="auto"
    )
  )
)


retry_agent = Agent(
  name="Retry agent",
  instructions="The user has not uploaded the required two documents for comparison. Suggest that they upload a total of two documents, using the paperclip icon.",
  model="gpt-5-nano",
  model_settings=ModelSettings(
    store=True,
    reasoning=Reasoning(
      effort="minimal",
      summary="auto"
    )
  )
)


provide_explanation = Agent(
  name="Provide explanation",
  instructions="Use the information in the uploaded documents to answer the user's question.",
  model="gpt-5-nano",
  model_settings=ModelSettings(
    store=True,
    reasoning=Reasoning(
      effort="minimal",
      summary="auto"
    )
  )
)


def approval_request(message: str):
  # TODO: Implement
  return True

class WorkflowInput(BaseModel):
  input_as_text: str


# Main code entrypoint
async def run_workflow(workflow_input: WorkflowInput):
  state = {

  }
  workflow = workflow_input.model_dump()
  conversation_history: list[TResponseInputItem] = [
    {
      "role": "user",
      "content": [
        {
          "type": "input_text",
          "text": workflow["input_as_text"]
        }
      ]
    }
  ]
  triage_request_result_temp = await Runner.run(
    triage_request,
    input=[
      *conversation_history
    ],
    run_config=RunConfig(trace_metadata={
      "__trace_source__": "agent-builder",
      "workflow_id": "wf_68e7d3ecffdc81909d6bd4ef54e13f97041f23f4d1d6d373"
    })
  )

  conversation_history.extend([item.to_input_item() for item in triage_request_result_temp.new_items])

  triage_request_result = {
    "output_text": triage_request_result_temp.final_output.json(),
    "output_parsed": triage_request_result_temp.final_output.model_dump()
  }
  if triage_request_result["output_parsed"]["classification"] == "compare":
    propose_reconciliation_result_temp = await Runner.run(
      propose_reconciliation,
      input=[
        *conversation_history
      ],
      run_config=RunConfig(trace_metadata={
        "__trace_source__": "agent-builder",
        "workflow_id": "wf_68e7d3ecffdc81909d6bd4ef54e13f97041f23f4d1d6d373"
      })
    )

    conversation_history.extend([item.to_input_item() for item in propose_reconciliation_result_temp.new_items])

    propose_reconciliation_result = {
      "output_text": propose_reconciliation_result_temp.final_output_as(str)
    }
    approval_message = f"Please review the proposal {propose_reconciliation_result["output_text"]}"

    if approval_request(approval_message):
        approval_agent_result_temp = await Runner.run(
          approval_agent,
          input=[
            *conversation_history
          ],
          run_config=RunConfig(trace_metadata={
            "__trace_source__": "agent-builder",
            "workflow_id": "wf_68e7d3ecffdc81909d6bd4ef54e13f97041f23f4d1d6d373"
          })
        )

        conversation_history.extend([item.to_input_item() for item in approval_agent_result_temp.new_items])

        approval_agent_result = {
          "output_text": approval_agent_result_temp.final_output.json(),
          "output_parsed": approval_agent_result_temp.final_output.model_dump()
        }
    else:
        rejection_agent_result_temp = await Runner.run(
          rejection_agent,
          input=[
            *conversation_history
          ],
          run_config=RunConfig(trace_metadata={
            "__trace_source__": "agent-builder",
            "workflow_id": "wf_68e7d3ecffdc81909d6bd4ef54e13f97041f23f4d1d6d373"
          })
        )

        conversation_history.extend([item.to_input_item() for item in rejection_agent_result_temp.new_items])

        rejection_agent_result = {
          "output_text": rejection_agent_result_temp.final_output_as(str)
        }
  elif triage_request_result["output_parsed"]["classification"] == "answer_question":
    provide_explanation_result_temp = await Runner.run(
      provide_explanation,
      input=[
        *conversation_history
      ],
      run_config=RunConfig(trace_metadata={
        "__trace_source__": "agent-builder",
        "workflow_id": "wf_68e7d3ecffdc81909d6bd4ef54e13f97041f23f4d1d6d373"
      })
    )

    conversation_history.extend([item.to_input_item() for item in provide_explanation_result_temp.new_items])

    provide_explanation_result = {
      "output_text": provide_explanation_result_temp.final_output_as(str)
    }
  else:
    retry_agent_result_temp = await Runner.run(
      retry_agent,
      input=[
        *conversation_history
      ],
      run_config=RunConfig(trace_metadata={
        "__trace_source__": "agent-builder",
        "workflow_id": "wf_68e7d3ecffdc81909d6bd4ef54e13f97041f23f4d1d6d373"
      })
    )

    conversation_history.extend([item.to_input_item() for item in retry_agent_result_temp.new_items])

    retry_agent_result = {
      "output_text": retry_agent_result_temp.final_output_as(str)
    }


In [34]:
# === Cell 3: AgentKit-only runner — text-or-files, approvals, safe patch, final output ===
# Edit only these two lines:
from typing import List, Dict, Any
USER_TEXT: str = ""  # "" if you only want to send files
FILE_PATHS: List[str] = ["/content/lattes.pdf", "/content/linkedin.pdf"]  # [] for text-only

import asyncio
from pathlib import Path

# Expect these from Cell 2 (we DO NOT modify Cell 2)
WorkflowInput = globals().get("WorkflowInput")
run_workflow = globals().get("run_workflow")
if WorkflowInput is None or run_workflow is None:
    raise RuntimeError("Cell 2 must define WorkflowInput and run_workflow(workflow_input).")

# Notebook loop helper (optional)
try:
    import nest_asyncio
    nest_asyncio.apply()
except Exception:
    pass

# ---- Files: upload with purpose="user_data" (Agents/Responses pattern) ----
# Docs: pass files as input items (type='input_file') in the same user turn.
# https://platform.openai.com/docs/api-reference/files
# https://platform.openai.com/docs/guides/pdf-files
from openai import OpenAI
_client = OpenAI()

def _upload_file_get_id(path: str) -> str:
    p = Path(path).expanduser().resolve()
    if not p.exists() or not p.is_file():
        raise FileNotFoundError(f"File not found: {p}")
    with p.open("rb") as f:
        obj = _client.files.create(file=f, purpose="user_data")
    return obj.id

def _build_file_items(paths: List[str]) -> List[Dict[str, Any]]:
    items: List[Dict[str, Any]] = []
    for p in (paths or []):
        try:
            fid = _upload_file_get_id(p)
            items.append({"type": "input_file", "transfer_method": "reference", "file_id": fid})
        except Exception as e:
            print(f"[WARN] Failed to upload {p}: {e}")
    return items

_FILE_ITEMS = _build_file_items(FILE_PATHS)

# ---- Approval gate for your flows (blocking y/N) ----
def approval_request(message: str) -> bool:
    print("\n=== Approval Required ===")
    print(message)
    ans = input("Approve? [y/N]: ").strip().lower()
    if ans in ("y", "yes"):
        return True
    reason = input("Rejection reason (optional): ").strip()
    if reason:
        print(f"Rejection reason noted: {reason}")
    return False

globals()["approval_request"] = approval_request  # visible for Cell 2

# ---- Hosted MCP approval (official Agents SDK pattern) ----
# https://openai.github.io/openai-agents-python/mcp/
try:
    from agents import Agent
    from agents.tool import HostedMCPTool, MCPToolApprovalRequest
except Exception:
    HostedMCPTool = None
    MCPToolApprovalRequest = None
    Agent = None

def _attach_mcp_approvals():
    if HostedMCPTool is None or Agent is None:
        return
    agents = {k: v for k, v in globals().items() if isinstance(v, Agent)}
    seen, stack = set(), list(agents.values())
    while stack:
        a = stack.pop()
        if id(a) in seen:
            continue
        seen.add(id(a))
        for t in getattr(a, "tools", []) or []:
            if isinstance(t, HostedMCPTool) and getattr(t, "on_approval_request", None) is None:
                def _on_approval(req: MCPToolApprovalRequest):
                    try:
                        server = req.data.server_label or getattr(req.data, "server_name", "")
                    except Exception:
                        server = ""
                    try:
                        tool_name = req.data.tool.name if req.data and req.data.tool else req.data.name
                    except Exception:
                        tool_name = getattr(req.data, "name", "<unknown>")
                    try:
                        args = getattr(req.data, "arguments", None) or getattr(req.data, "call", {}).get("arguments")
                    except Exception:
                        args = None
                    lines = ["MCP tool call requires approval:",
                             f" • Server: {server}",
                             f" • Tool:   {tool_name}"]
                    if args is not None:
                        lines.append(f" • Args:   {args}")
                    return {"approve": approval_request("\n".join(lines))}
                t.on_approval_request = _on_approval
        for h in getattr(a, "handoffs", []) or []:
            stack.append(h)

_attach_mcp_approvals()

# ---- Safe, idempotent patch of Runner.run (no nonlocal!) ----
# Runner.run accepts a string OR a list of Responses input items. We:
# 1) ensure the FIRST user turn has input_text + our input_file items (once),
# 2) capture the LAST final_output to print at the end.
# https://openai.github.io/openai-agents-python/running_agents/
from agents import Runner as _Runner

# Guard: only patch once
if not getattr(_Runner, "_cell3_patched", False):
    _Runner._original_run = _Runner.run             # save original
    _Runner._files_injected_once = False            # flag lives on the class (no nonlocal)
    _Runner._last_final_output = {"agent": None, "obj": None}

    async def _patched_run(agent, *args, **kwargs):
        # Inject files/text into the first user turn only on the very first call
        if not _Runner._files_injected_once:
            input_items = kwargs.get("input", None)
            if isinstance(input_items, list) and input_items:
                for msg in input_items:
                    if isinstance(msg, dict) and msg.get("role") == "user" and isinstance(msg.get("content"), list):
                        has_text = any(isinstance(it, dict) and it.get("type") == "input_text" for it in msg["content"])
                        if not has_text:
                            msg["content"].insert(0, {"type": "input_text", "text": USER_TEXT or ""})
                        if _FILE_ITEMS:
                            msg["content"].extend(_FILE_ITEMS)
                        _Runner._files_injected_once = True
                        break
            elif isinstance(input_items, str) or input_items is None:
                text = input_items if isinstance(input_items, str) else (USER_TEXT or "")
                kwargs["input"] = [{"role": "user", "content": [{"type": "input_text", "text": text}] + list(_FILE_ITEMS)}]
                _Runner._files_injected_once = True

        # Call original Runner.run
        res = await _Runner._original_run(agent, *args, **kwargs)

        # Capture final output for reporting
        _Runner._last_final_output["agent"] = getattr(agent, "name", "<agent>")
        _Runner._last_final_output["obj"] = getattr(res, "final_output", None)
        return res

    _Runner.run = _patched_run
    _Runner._cell3_patched = True

# ---- Execute your workflow exactly as Cell 2 expects ----
workflow_input = WorkflowInput(input_as_text=USER_TEXT)

async def _main():
    return await run_workflow(workflow_input)

try:
    loop = asyncio.get_event_loop()
    RUN_RESULT = loop.run_until_complete(_main())
except RuntimeError:
    RUN_RESULT = asyncio.run(_main())

# ---- Always print a useful final result ----
def _final_output_to_text(final_obj) -> str:
    if final_obj is None:
        return ""
    try:
        if isinstance(final_obj, str):
            return final_obj
        if hasattr(final_obj, "model_dump_json"):
            return final_obj.model_dump_json(indent=2, ensure_ascii=False)
        if hasattr(final_obj, "json"):
            return final_obj.json(indent=2, ensure_ascii=False)
        if hasattr(final_obj, "model_dump"):
            import json as _json
            return _json.dumps(final_obj.model_dump(), ensure_ascii=False, indent=2)
        return str(final_obj)
    except Exception:
        return str(final_obj)

print("\n=== Workflow finished ===")
fo = getattr(_Runner, "_last_final_output", {}).get("obj") if hasattr(_Runner, "_last_final_output") else None
if fo is not None:
    print(f"--- Final output (last agent: {_Runner._last_final_output.get('agent')}) ---")
    print(_final_output_to_text(fo))
else:
    print(RUN_RESULT if RUN_RESULT is not None else "<no final_output and run_workflow returned None>")

# Optional: unpatch to keep notebook state clean if you re-run the cell
try:
    if getattr(_Runner, "_cell3_patched", False):
        _Runner.run = _Runner._original_run
        delattr(_Runner, "_original_run")
        delattr(_Runner, "_cell3_patched")
        delattr(_Runner, "_files_injected_once")
        delattr(_Runner, "_last_final_output")
except Exception:
    pass


RecursionError: maximum recursion depth exceeded